In [ ]:
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [ ]:
fake = pd.read_csv("/content/True_News.csv")
real = pd.read_csv("/content/Fake_news.csv")

fake["label"] = 0
real["label"] = 1

data = pd.concat([fake, real])
data = data.sample(frac=1).reset_index(drop=True)

data = data[["text", "label"]]
data.head()

/tmp/ipython-input-2245955042.py:2: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171) have mixed types. Specify dtype option on import or set low_memory=False.
  real = pd.read_csv("/content/Fake_news.csv")


,text,label
0,"After watching this video, and seeing a side o...",1
1,BEIRUT (Reuters) - The International Support G...,0
2,Donald Trump has made a fine art of p*ssing of...,1
3,TAIPEI (Reuters) - Taiwan has suspended refine...,0
4,George W. Bush was famously attacked with flyi...,1


In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    return text

data["text"] = data["text"].apply(clean_text)

In [ ]:
X = data["text"]
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
vocab_size = 5000
max_length = 200

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

In [ ]:
from tensorflow.keras.layers import Input

model = Sequential()


model.add(Input(shape=(max_length,)))

model.add(Embedding(input_dim=vocab_size, output_dim=128))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 689,473 (2.63 MB)

 Trainable params: 689,473 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 103s 223ms/step - accuracy: 0.8895 - loss: 0.2714 - val_accuracy: 0.9698 - val_loss: 0.0919
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 105s 233ms/step - accuracy: 0.9573 - loss: 0.1399 - val_accuracy: 0.9751 - val_loss: 0.0759
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 101s 226ms/step - accuracy: 0.9801 - loss: 0.0658 - val_accuracy: 0.9713 - val_loss: 0.0898
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 103s 229ms/step - accuracy: 0.9708 - loss: 0.0858 - val_accuracy: 0.9759 - val_loss: 0.0816
Epoch 5/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 101s 226ms/step - accuracy: 0.9807 - loss: 0.0559 - val_accuracy: 0.9627 - val_loss: 0.1071


In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", accuracy)

281/281 ━━━━━━━━━━━━━━━━━━━━ 10s 36ms/step - accuracy: 0.9681 - loss: 0.1043
Test Accuracy: 0.9666072726249695


In [ ]:
def predict_news(text):
    text = clean_text(text)
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_length)
    prediction = model.predict(pad)

    if prediction > 0.5:
        print("Real News")
    else:
        print("Fake News")

predict_news("President signs new economic policy today")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step
Real News


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_news(text):
    # Clean text
    text = clean_text(text)

    # Convert to sequence
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=max_length)

    # Prediction
    prediction = model.predict(pad)[0][0]

    if prediction > 0.5:
        return f"Real News (Confidence: {round(prediction*100,2)}%)"
    else:
        return f"Fake News (Confidence: {round((1-prediction)*100,2)}%)"

# Create Interface
app = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=6, placeholder="Enter news text here..."),
    outputs="text",
    title="Fake News Detector (LSTM Model)",
    description="This model detects whether the news is Fake or Real using Deep Learning (LSTM)."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94affe76b2dab7be0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
Running on public URL: https://xxxxx.gradio.live